<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


# Название проекта: Система управления задачами (Task Management System)
### Вариант задания: 23
В рамках **Спринта 1 (Вариант 23)** разработана объектно-ориентированная система управления задачами на языке C#. Проект демонстрирует фундаментальные концепции ООП, правильное проектирование классов и их взаимодействие.

#### Ключевые компоненты и реализация:
1. **Базовый класс `Task`**:
   * **Инкапсуляция и валидация данных**: Закрытые поля (`private _priority` и др.) и свойства с логикой проверки. Свойство `Priority` принимает числовые значения только от 1 до 5; при передаче некорректного значения выбрасывается исключение `ArgumentOutOfRangeException`.
   * **Статический счётчик**: Переменная `static int TotalTasks` подчитывает общее количество созданных задач в системе через вызов конструктора.

2. **Производные классы (Наследование и Полиморфизм)**:
   * **`DelegateTask`**: Расширяет класс `Task` полем `DueDate` (дата сдачи) и переопределяет метод `MarkAsComplete()` через `override`.
   * **`TeamTask`**: Добавляет свойство `TeamName` и переопределяет метод `ReassignTo()` для вывода информации о назначении исполнителя в команду.
   * **`ResearchTask`**: Добавляет свойство `DataSource` и расширяет вывод метода `GetTaskDetails()`, задействуя родительскую реализацию через `base.GetTaskDetails()`.

3. **Взаимодействие объектов**:
   * **Класс `ProjectManager`**: Служит контроллером, метод которого `ReviewAndComplete(Task task)` принимает объекты любых типа-наследников `Task` и вызывает их переопределенные методы, демонстрируя полиморфизм на практике.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;

// 1. БАЗОВЫЙ КЛАСС TASK
public class Task
{
    private int _taskId;
    private string _taskName;
    private int _priority;
    private string _assignee;

    public int TaskId
    {
        get { return _taskId; }
        set { _taskId = value; }
    }

    public string TaskName
    {
        get { return _taskName; }
        set { _taskName = value; }
    }

    public int Priority
    {
        get { return _priority; } 
        set
        {
            if (value >= 1 && value <= 5)
                _priority = value;
            else
                throw new ArgumentOutOfRangeException("Приоритет должен быть от 1 до 5!");
        }
    }

    public string Assignee
    {
        get { return _assignee; }
        set { _assignee = value; }
    }

    public bool IsCompleted { get; protected set; } //статус выполнения

    public static int TotalTasks { get; private set; } = 0; //статис счетчик

    // Конструктор
    public Task(int taskId, string taskName, int priority, string assignee)
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority; // от 1 до 5
        Assignee = assignee;
        IsCompleted = false;
        TotalTasks++; // ++++ общ счет
    }

    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Console.WriteLine($"[Задача #{TaskId}] '{TaskName}' отмечена как выполненная.");
    }

    public virtual string GetTaskDetails()
    {
        return $"ID: {TaskId} | Задача: {TaskName} | Приоритет: {Priority} | Исполнитель: {Assignee} | Статус: {(IsCompleted ? "Завершено" : "В работе")}";
    }

    public virtual void ReassignTo(string newAssignee)
    {
        string oldAssignee = Assignee;
        Assignee = newAssignee;
        Console.WriteLine($"[Задача #{TaskId}] Переназначена с {oldAssignee} на {newAssignee}.");
    }
}

public class DelegateTask : Task
{
    public DateTime DueDate { get; set; } //дата сдачи

    public DelegateTask(int taskId, string taskName, int priority, string assignee, DateTime dueDate)
        : base(taskId, taskName, priority, assignee)
    {
        DueDate = dueDate;
    }

    public override void MarkAsComplete()
    {
        IsCompleted = true;
        Console.WriteLine($"[Делегированная задача #{TaskId}] '{TaskName}' выполнена к {DueDate.ToString("dd.MM.yyyy")}.");
    }
}

public class TeamTask : Task
{
    public string TeamName { get; set; }

    public TeamTask(int taskId, string taskName, int priority, string assignee, string teamName)
        : base(taskId, taskName, priority, assignee)
    {
        TeamName = teamName;
    }

    public override void ReassignTo(string newAssignee)
    {
        Assignee = newAssignee;
        Console.WriteLine($"[Командная задача #{TaskId}] Новый участник '{newAssignee}' назначен в команду '{TeamName}' для этой задачи.");
    }
}

public class ResearchTask : Task
{
    public string DataSource { get; set; }

    public ResearchTask(int taskId, string taskName, int priority, string assignee, string dataSource)
        : base(taskId, taskName, priority, assignee)
    {
        DataSource = dataSource;
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() + $" | Источник данных: {DataSource}";
    }
}

public class ProjectManager
{
    public string Name { get; set; }

    public ProjectManager(string name)
    {
        Name = name;
    }

    public void ReviewAndComplete(Task task)
    {
        Console.WriteLine($"\nМенеджер {Name} проверяет задачу: {task.GetTaskDetails()}");
        task.MarkAsComplete();
    }
}


// Выбираем имя менеджера
ProjectManager manager = new ProjectManager("Алексей");

// Создаем задачи (ID, Название задачи, Приоритет, Исполнитель, Спец-параметр)
DelegateTask delTask = new DelegateTask(101, "Подготовить отчёт", 4, "Иван", DateTime.Now.AddDays(3));
TeamTask teamTask = new TeamTask(102, "Разработка интерфейса", 5, "Елена", "Команда разработки");
ResearchTask resTask = new ResearchTask(103, "Анализ рынка", 3, "Пётр", "Внутренняя база данных");

Console.WriteLine($"Всего задач создано: {Task.TotalTasks}\n");

teamTask.ReassignTo("Максим");
manager.ReviewAndComplete(delTask);
manager.ReviewAndComplete(resTask);

The below script needs to be able to find the current output cell; this is an easy method to get it.

Всего задач создано: 3

[Командная задача #102] Новый участник 'Максим' назначен в команду 'Команда разработки' для этой задачи.

Менеджер Алексей проверяет задачу: ID: 101 | Задача: Подготовить отчёт | Приоритет: 4 | Исполнитель: Иван | Статус: В работе
[Делегированная задача #101] 'Подготовить отчёт' выполнена к 24.09.2026.

Менеджер Алексей проверяет задачу: ID: 103 | Задача: Анализ рынка | Приоритет: 3 | Исполнитель: Пётр | Статус: В работе | Источник данных: Внутренняя база данных
[Задача #103] 'Анализ рынка' отмечена как выполненная.


: 